# 04 - Evaluation and Risk Decisions
Loads the saved model, scores the saved test set, and turns probabilities into Low / Medium / High risk decisions using the thresholds in `src/config.py`.

In [1]:
import os, sys
sys.path.insert(0, os.path.abspath(".."))   # so we can import from src/

import joblib
import numpy as np
import pandas as pd
from src import config
from src.data_preprocessing import load_processed_data

model = joblib.load(config.MODEL_PATH)
_, X_test, _, y_test = load_processed_data()
y_pred_prob = model.predict_proba(X_test)[:, 1]
print("Thresholds -> Low <", config.LOW_RISK_MAX, "| High >=", config.HIGH_RISK_MIN)

Thresholds -> Low < 0.3 | High >= 0.6


In [2]:
results = X_test.copy()
results["actual_default"] = y_test.values
results["default_probability"] = y_pred_prob
results["decision"] = results["default_probability"].apply(config.risk_band)
results[["actual_default", "default_probability", "decision"]].head()

,actual_default,default_probability,decision
0,0,0.329355,Medium Risk - Manual Review
1,0,0.223792,Low Risk - Approve
2,0,0.471120,Medium Risk - Manual Review
3,1,0.257689,Low Risk - Approve
4,0,0.089612,Low Risk - Approve


In [3]:
results["decision"].value_counts(normalize=True) * 100

decision
Medium Risk - Manual Review    56.316667
Low Risk - Approve             22.533333
High Risk - Reject             21.150000
Name: proportion, dtype: float64

In [4]:
approval_rate = (results["decision"] == "Low Risk - Approve").mean()
rejection_rate = (results["decision"] == "High Risk - Reject").mean()
print(f"Approval Rate : {approval_rate*100:.2f}%")
print(f"Rejection Rate: {rejection_rate*100:.2f}%")

Approval Rate : 22.53%
Rejection Rate: 21.15%


### Actual default rate inside each risk band
A good model has a low default rate in *Low Risk* and a high one in *High Risk*.

In [5]:
results.groupby("decision")["actual_default"].agg(["count", "mean"]).rename(columns={"mean": "actual_default_rate"})

,count,actual_default_rate
decision,,
High Risk - Reject,1269,0.491726
Low Risk - Approve,1352,0.119083
Medium Risk - Manual Review,3379,0.160402


### Default recall at different thresholds

In [6]:
from sklearn.metrics import recall_score, precision_score

for t in np.arange(0.2, 0.8, 0.05):
    preds = (y_pred_prob >= t).astype(int)
    print(f"Threshold {t:.2f} -> Recall: {recall_score(y_test, preds):.2f}  Precision: {precision_score(y_test, preds):.2f}")

Threshold 0.20 -> Recall: 0.94  Precision: 0.23
Threshold 0.25 -> Recall: 0.91  Precision: 0.24
Threshold 0.30 -> Recall: 0.88  Precision: 0.25
Threshold 0.35 -> Recall: 0.82  Precision: 0.26
Threshold 0.40 -> Recall: 0.74  Precision: 0.27
Threshold 0.45 -> Recall: 0.68  Precision: 0.31


Threshold 0.50 -> Recall: 0.60  Precision: 0.38
Threshold 0.55 -> Recall: 0.52  Precision: 0.45
Threshold 0.60 -> Recall: 0.47  Precision: 0.49
Threshold 0.65 -> Recall: 0.41  Precision: 0.55
Threshold 0.70 -> Recall: 0.35  Precision: 0.60
Threshold 0.75 -> Recall: 0.27  Precision: 0.63
Threshold 0.80 -> Recall: 0.19  Precision: 0.66
